# Hinglish sentiment research
Runs the reproducible CLI in `.venv-research`; run setup first. This uses a custom split, not the exact reference-paper corpus or official SentiMix test split. Completed runs are preserved.

In [ ]:
from pathlib import Path
import json, os, subprocess
ROOT = Path.cwd()
while not (ROOT / "run_research.py").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
assert (ROOT / "run_research.py").exists(), "Open the notebook inside the PBL project"
PYTHON = ROOT / ".venv-research/Scripts/python.exe"
CONFIG = ROOT / "configs/research.json"
settings = json.loads(CONFIG.read_text())
def run(*arguments):
    command = [str(PYTHON), "-u", str(ROOT / "run_research.py"), *arguments, "--config", str(CONFIG)]
    env = {**os.environ, "PYTHONIOENCODING": "utf-8"}
    with subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", env=env) as process:
        for line in process.stdout:
            print(line, end="")
        if process.wait():
            raise RuntimeError("Research command failed; inspect the output above")
run("doctor")


## Freeze data once
Review the recorded exclusions and class counts.

In [ ]:
audit = ROOT / settings["prepared_dir"] / "audit.json"
if not audit.exists():
    run("prepare")
print(audit.read_text())


## Train baselines
Failed runs are intentionally not overwritten; use a new output directory to restart.

In [ ]:
for model in ["naive_bayes", "svm"]:
    complete = ROOT / settings["output_dir"] / model / "seed_42/complete.json"
    if not complete.exists():
        run("train", "--models", model)


## Fine-tune mBERT
This runs all three epochs on the full training split.

In [ ]:
complete = ROOT / settings["output_dir"] / "mbert/seed_42/complete.json"
if not complete.exists():
    run("train", "--models", "mbert", "--seeds", "42")
# Optional additional training seeds, using the identical frozen split:
# run("train", "--models", "mbert", "--seeds", "43", "44")


## Compare completed runs

In [ ]:
run("report")
from IPython.display import Markdown, Image, display
output = ROOT / settings["output_dir"]
display(Markdown((output / "results.md").read_text()))
for model in ["naive_bayes", "svm", "mbert"]:
    figure = output / model / "seed_42/test_confusion_matrix.png"
    if figure.exists():
        display(Image(filename=str(figure), width=600))
